# Experiment: Results defence audit

Objective:
- Audit the current Jupyter notebook stack against the dissertation purpose, research question, and defended leakage-aware benchmarking scope.
- Reproduce compact thesis-facing diagnostics from the preserved result archives without rerunning heavy models.
- Verify whether the realised feature space supports claims about data domains, especially the distinction between intended multimodal scope and preserved executable evidence.


In [1]:
# Setup: imports, display options, and repository paths
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(obj):
        print(obj)

pd.set_option('display.max_colwidth', 160)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'AGENTS.md').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


REPO_ROOT = find_repo_root(Path.cwd())
RESULTS_DIR = REPO_ROOT / 'jupyter_notebooks' / 'data' / 'out' / 'results_models'
DISCUSSION_TABLE_DIR = REPO_ROOT / 'output' / 'doc' / 'v4_6_results_discussion_pack_2026_05_03' / '04_tables'
OUTPUT_NOTEBOOK = REPO_ROOT / 'output' / 'jupyter-notebook' / 'results_defence_audit.ipynb'

{
    'repo_root': str(REPO_ROOT),
    'results_dir_exists': RESULTS_DIR.exists(),
    'discussion_table_dir_exists': DISCUSSION_TABLE_DIR.exists(),
    'this_notebook': str(OUTPUT_NOTEBOOK),
}


{'repo_root': 'c:\\Users\\vasil\\Documents\\Mestrado\\dissertação mestrado\\crypto_thesis',
 'results_dir_exists': True,
 'discussion_table_dir_exists': True,
 'this_notebook': 'c:\\Users\\vasil\\Documents\\Mestrado\\dissertação mestrado\\crypto_thesis\\output\\jupyter-notebook\\results_defence_audit.ipynb'}

## Plan

- Inventory the active and duplicate notebooks.
- Verify whether the current notebook chain already supports the dissertation's defended empirical claims.
- Measure split comparability and benchmark-relative performance from the preserved result files.
- Check the realised feature-domain coverage used by the saved backtest archive.
- Decide whether another notebook is needed and record the rationale.


In [2]:
# Inventory the notebook stack and identify duplicate copies
SCAN_SPECS = [
    ('repo_root', REPO_ROOT.glob('*.ipynb')),
    ('jupyter_notebooks', (REPO_ROOT / 'jupyter_notebooks').glob('**/*.ipynb')),
    ('.jupyter', (REPO_ROOT / '.jupyter').glob('**/*.ipynb')),
]


def sha256_short(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()[:12]


rows = []
for location, iterator in SCAN_SPECS:
    for path in iterator:
        rows.append(
            {
                'location': location,
                'relative_path': path.relative_to(REPO_ROOT).as_posix(),
                'name': path.name,
                'size_kb': round(path.stat().st_size / 1024, 1),
                'sha12': sha256_short(path),
            }
        )

inventory = pd.DataFrame(rows).sort_values(['name', 'location', 'relative_path']).reset_index(drop=True)
inventory['same_hash_copies'] = inventory.groupby('sha12')['sha12'].transform('size')
inventory['same_name_copies'] = inventory.groupby('name')['name'].transform('size')

inventory_summary = (
    inventory.groupby('location')
    .agg(notebooks=('name', 'size'), distinct_hashes=('sha12', 'nunique'))
    .reset_index()
)

v1_inventory = inventory[inventory['name'].str.startswith('v1_')].copy()
duplicate_v1 = v1_inventory[v1_inventory['same_name_copies'] > 1].copy()

display(inventory_summary)
display(v1_inventory[['location', 'relative_path', 'size_kb', 'sha12', 'same_hash_copies', 'same_name_copies']])
display(duplicate_v1[['relative_path', 'sha12']])


,location,notebooks,distinct_hashes
0,.jupyter,10,10
1,jupyter_notebooks,19,19
2,repo_root,5,5


,location,relative_path,size_kb,sha12,same_hash_copies,same_name_copies
25,jupyter_notebooks,jupyter_notebooks/v1_etl_process.ipynb,23.2,ac3700d96a29,1,2
26,repo_root,v1_etl_process.ipynb,22.8,fc08099efbfa,1,2
27,jupyter_notebooks,jupyter_notebooks/v1_integrity_EDA.ipynb,283.1,d2f68ee2f415,1,2
28,repo_root,v1_integrity_EDA.ipynb,280.6,8775970ce525,1,2
29,repo_root,v1_models_test-Copy1.ipynb,55.2,0e0723470c17,1,1
30,jupyter_notebooks,jupyter_notebooks/v1_models_test.ipynb,198.5,ec5ee2880d38,1,2
31,repo_root,v1_models_test.ipynb,75.7,279fef1dc20d,1,2
32,jupyter_notebooks,jupyter_notebooks/v1_result_analisys.ipynb,9.9,2314883ef006,1,2
33,repo_root,v1_result_analisys.ipynb,9.6,3f97aa865990,1,2


,relative_path,sha12
25,jupyter_notebooks/v1_etl_process.ipynb,ac3700d96a29
26,v1_etl_process.ipynb,fc08099efbfa
27,jupyter_notebooks/v1_integrity_EDA.ipynb,d2f68ee2f415
28,v1_integrity_EDA.ipynb,8775970ce525
30,jupyter_notebooks/v1_models_test.ipynb,ec5ee2880d38
31,v1_models_test.ipynb,279fef1dc20d
32,jupyter_notebooks/v1_result_analisys.ipynb,2314883ef006
33,v1_result_analisys.ipynb,3f97aa865990


In [3]:

# Diagnose the thesis-facing status of the main notebook chain

def notebook_source(path: Path) -> str:
    nb = json.loads(path.read_text(encoding='utf-8'))
    return "\n".join(''.join(cell.get('source', [])) for cell in nb.get('cells', []))


root_v1_paths = {
    'v1_etl_process.ipynb': REPO_ROOT / 'v1_etl_process.ipynb',
    'v1_integrity_EDA.ipynb': REPO_ROOT / 'v1_integrity_EDA.ipynb',
    'v1_models_test.ipynb': REPO_ROOT / 'v1_models_test.ipynb',
    'v1_result_analisys.ipynb': REPO_ROOT / 'v1_result_analisys.ipynb',
}

notebook_status = []
for name, path in root_v1_paths.items():
    src = notebook_source(path) if path.exists() else ''
    notebook_status.append(
        {
            'notebook': name,
            'exists': path.exists(),
            'contains_results_models_path': 'results_models' in src,
            'contains_legacy_results_path': 'data/out/results' in src,
            'contains_sentiment_keyword': 'sentiment' in src.lower(),
            'contains_hash_keyword': 'hash' in src.lower(),
        }
    )

notebook_status = pd.DataFrame(notebook_status)

thesis_chain_recommendation = pd.DataFrame(
    [
        {
            'stage': 'ETL',
            'notebook': 'v1_etl_process.ipynb',
            'recommended_status': 'retain',
            'rationale': 'This is the relevant source for data assembly, lagging logic, and claims about intended sentiment/hash inputs.',
        },
        {
            'stage': 'Integrity / EDA',
            'notebook': 'v1_integrity_EDA.ipynb',
            'recommended_status': 'retain',
            'rationale': 'This remains the main notebook for descriptive checks and sanity validation of the hourly dataset.',
        },
        {
            'stage': 'Backtesting',
            'notebook': 'v1_models_test.ipynb',
            'recommended_status': 'retain with duplicate-control',
            'rationale': 'The modelling evidence exists, but duplicate copies across folders create avoidable reproducibility risk.',
        },
        {
            'stage': 'Results interpretation',
            'notebook': 'v1_result_analisys.ipynb',
            'recommended_status': 'supersede for thesis defence',
            'rationale': 'The notebook still targets the legacy path data/out/results instead of the preserved results_models archive.',
        },
        {
            'stage': 'Results defence bridge',
            'notebook': 'output/jupyter-notebook/results_defence_audit.ipynb',
            'recommended_status': 'new canonical audit layer',
            'rationale': 'Use this notebook to connect the preserved archives to the dissertation purpose, research question, and cautious discussion framing.',
        },
    ]
)

display(notebook_status)
display(thesis_chain_recommendation)


,notebook,exists,contains_results_models_path,contains_legacy_results_path,contains_sentiment_keyword,contains_hash_keyword
0,v1_etl_process.ipynb,True,False,False,True,True
1,v1_integrity_EDA.ipynb,True,False,False,False,False
2,v1_models_test.ipynb,True,True,True,False,False
3,v1_result_analisys.ipynb,True,False,True,False,False


,stage,notebook,recommended_status,rationale
0,ETL,v1_etl_process.ipynb,retain,"This is the relevant source for data assembly, lagging logic, and claims about intended sentiment/hash inputs."
1,Integrity / EDA,v1_integrity_EDA.ipynb,retain,This remains the main notebook for descriptive checks and sanity validation of the hourly dataset.
2,Backtesting,v1_models_test.ipynb,retain with duplicate-control,"The modelling evidence exists, but duplicate copies across folders create avoidable reproducibility risk."
3,Results interpretation,v1_result_analisys.ipynb,supersede for thesis defence,The notebook still targets the legacy path data/out/results instead of the preserved results_models archive.
4,Results defence bridge,output/jupyter-notebook/results_defence_audit.ipynb,new canonical audit layer,"Use this notebook to connect the preserved archives to the dissertation purpose, research question, and cautious discussion framing."


## Thesis-facing evidence from the preserved results archive

The dissertation is currently defended as a leakage-aware benchmarking study rather than as proof of stable multimodal or hybrid superiority. The next cells therefore verify whether the saved archives support that narrower but defensible claim.


In [4]:
# Load summary and per-split result archives
summary = pd.read_csv(RESULTS_DIR / 'summary_metrics_by_H.csv')
per_split = pd.read_csv(RESULTS_DIR / 'per_split_metrics.csv')

split_counts = per_split.groupby(['H', 'Model']).size().rename('n_splits').reset_index()

dm_profile = (
    per_split.assign(
        dm_better=lambda df: ((df['DM_p_vsNaive0'] < 0.05) & (df['DM_stat_vsNaive0'] < 0)).astype(int),
        dm_worse=lambda df: ((df['DM_p_vsNaive0'] < 0.05) & (df['DM_stat_vsNaive0'] > 0)).astype(int),
    )
    .groupby(['H', 'Model'])[['dm_better', 'dm_worse']]
    .sum()
    .reset_index()
    .merge(split_counts, on=['H', 'Model'], how='left')
)
dm_profile['dm_inconclusive'] = dm_profile['n_splits'] - dm_profile['dm_better'] - dm_profile['dm_worse']
dm_profile['dm_better_rate'] = dm_profile['dm_better'] / dm_profile['n_splits']
dm_profile['dm_worse_rate'] = dm_profile['dm_worse'] / dm_profile['n_splits']

naive0_mae = summary.loc[summary['Model'] == 'Naive0', ['H', 'MAE']].rename(columns={'MAE': 'MAE_naive0'})
benchmark_table = (
    summary.merge(split_counts, on=['H', 'Model'], how='left')
    .merge(naive0_mae, on='H', how='left')
    .merge(dm_profile[['H', 'Model', 'dm_better', 'dm_worse', 'dm_inconclusive', 'dm_better_rate', 'dm_worse_rate']], on=['H', 'Model'], how='left')
    .assign(mae_gap_vs_naive0=lambda df: df['MAE'] - df['MAE_naive0'])
    .sort_values(['H', 'MAE', 'Model'])
    .reset_index(drop=True)
)

best_model_by_h = benchmark_table.groupby('H', as_index=False).first()[['H', 'Model', 'MAE', 'mae_gap_vs_naive0', 'n_splits']]
best_non_naive_by_h = (
    benchmark_table[~benchmark_table['Model'].isin(['Naive0', 'NaiveLast'])]
    .groupby('H', as_index=False)
    .first()[['H', 'Model', 'MAE', 'mae_gap_vs_naive0', 'n_splits']]
)

display(benchmark_table[['H', 'Model', 'n_splits', 'MAE', 'mae_gap_vs_naive0', 'DA', 'dm_better', 'dm_worse', 'dm_better_rate', 'dm_worse_rate']])
display(best_model_by_h)
display(best_non_naive_by_h)


,H,Model,n_splits,MAE,mae_gap_vs_naive0,DA,dm_better,dm_worse,dm_better_rate,dm_worse_rate
0,1,Naive0,53,0.003627,0.000000,0.000112,0,0,0.000000,0.000000
1,1,OLS,53,0.003683,0.000056,0.513279,0,4,0.000000,0.075472
2,1,RF,53,0.004033,0.000405,0.511226,0,13,0.000000,0.245283
3,1,NaiveLast,53,0.005314,0.001687,0.493064,0,52,0.000000,0.981132
4,1,ARIMA,53,0.045971,0.042344,0.502673,0,53,0.000000,1.000000
5,6,Naive0,53,0.009110,0.000000,0.000225,0,0,0.000000,0.000000
6,6,OLS,53,0.009533,0.000423,0.525085,4,18,0.075472,0.339623
7,6,NaiveLast,53,0.010111,0.001000,0.487718,0,35,0.000000,0.660377
8,6,RF,53,0.010936,0.001826,0.503266,0,30,0.000000,0.566038
9,6,ARIMA,53,0.046870,0.037760,0.509717,0,52,0.000000,0.981132


,H,Model,MAE,mae_gap_vs_naive0,n_splits
0,1,Naive0,0.003627,0.0,53
1,6,Naive0,0.009110,0.0,53
2,24,Naive0,0.019720,0.0,53


,H,Model,MAE,mae_gap_vs_naive0,n_splits
0,1,OLS,0.003683,0.000056,53
1,6,OLS,0.009533,0.000423,53
2,24,OLS,0.020895,0.001175,53


In [5]:
# Quantify feature-domain coverage in the preserved backtest archive
feature_row = per_split.query("Model == 'OLS' and H == 1").iloc[0]
features = [feature.strip() for feature in str(feature_row['features']).split(',') if feature.strip()]

market_features = {'open', 'high', 'low', 'volume', 'quote_vol', 'trades', 'taker_buy_base', 'taker_buy_quote', 'ret_t'}
technical_prefixes = ('volume_', 'volatility_', 'trend_', 'momentum_', 'others_')
technical_exact = {'sma200', 'sma50', 'ema200', 'ema50', 'slope', 'slope_obv', 'slope_rsi', 'morningstar', 'hammer', 'piercing', '3soldiers', 'engulfing'}
sentiment_tokens = ('sentiment', 'twitter', 'reddit', 'news', 'fear', 'greed', 'bert', 'vader')
network_tokens = ('hash', 'difficulty', 'network', 'active', 'address', 'addresses', 'miner', 'fee', 'fees', 'onchain', 'transaction', 'mvrv', 'utxo', 'supply')


def classify_feature(name: str) -> str:
    lowered = name.lower()
    if lowered in market_features:
        return 'market / returns'
    if lowered.startswith(technical_prefixes) or lowered in technical_exact:
        return 'technical indicators'
    if any(token in lowered for token in sentiment_tokens):
        return 'sentiment'
    if any(token in lowered for token in network_tokens):
        return 'network / on-chain'
    return 'other / metadata'


feature_df = pd.DataFrame({'feature': features})
feature_df['domain'] = feature_df['feature'].map(classify_feature)

feature_domain_summary = (
    feature_df.groupby('domain')
    .agg(
        n_features=('feature', 'nunique'),
        example_features=('feature', lambda s: ', '.join(list(s)[:10]))
    )
    .reset_index()
    .sort_values('n_features', ascending=False)
)

keyword_terms = ['sentiment', 'hash', 'news', 'reddit', 'twitter', 'network']
keyword_rows = []
for name, path in root_v1_paths.items():
    src = notebook_source(path).lower()
    keyword_rows.append(
        {
            'notebook': name,
            **{term: int(term in src) for term in keyword_terms},
        }
    )

keyword_scan = pd.DataFrame(keyword_rows)

display(feature_domain_summary)
display(keyword_scan)


,domain,n_features,example_features
2,technical indicators,93,"volume_adi, volume_obv, volume_cmf, volume_fi, volume_em, volume_sma_em, volume_vpt, volume_vwap, volume_mfi, volume_nvi"
0,market / returns,9,"open, high, low, volume, quote_vol, trades, taker_buy_base, taker_buy_quote, ret_t"
1,other / metadata,1,Unnamed: 0


,notebook,sentiment,hash,news,reddit,twitter,network
0,v1_etl_process.ipynb,1,1,0,0,0,0
1,v1_integrity_EDA.ipynb,0,0,0,0,0,0
2,v1_models_test.ipynb,0,0,0,0,0,0
3,v1_result_analisys.ipynb,0,0,0,0,0,0


## Results

The notebook audit supports four main conclusions. First, the repository already contains an ETL-to-results chain that is broadly sufficient for the defended thesis, but duplicate copies across folders weaken reproducibility discipline. Second, the principal gap is not missing model classes but the absence of one clean notebook that translates the preserved result archive into dissertation-facing benchmark diagnostics. Third, the realised feature matrix preserved in the core backtest archive is dominated by market and technical variables, while sentiment and network-related inputs appear mainly as intended or partially handled ETL scope rather than as clearly preserved core-result features. Fourth, the saved summary metrics continue to support the dissertation's cautious answer to the research question: under the adopted leakage-aware design, the executed non-naive models do not establish stable superiority over the strongest naive benchmark across the 1-hour, 6-hour, and 24-hour horizons.


In [6]:
# Final thesis-alignment verdict for notebook planning
alignment_verdict = pd.DataFrame(
    [
        {
            'question': 'Does the current notebook stack already cover ETL, EDA, core modelling, and saved backtest outputs?',
            'answer': 'Yes',
            'evidence': 'The v1 ETL, integrity, modelling, and saved result archives are present in the repository.',
        },
        {
            'question': 'Is another heavy modelling notebook required to address the dissertation purpose?',
            'answer': 'No',
            'evidence': 'The main weakness is fragmented reproducibility and stale result-analysis paths, not an absence of core benchmark evidence.',
        },
        {
            'question': 'Is an additional consolidation notebook methodologically justified?',
            'answer': 'Yes',
            'evidence': 'This notebook replaces the thesis-defence role that the legacy result-analysis notebook no longer fulfils cleanly.',
        },
        {
            'question': 'Should the dissertation explicitly discuss realised feature-domain coverage?',
            'answer': 'Yes',
            'evidence': 'The preserved backtest feature matrix is overwhelmingly market/technical, so multimodal scope must be framed as partial rather than complete.',
        },
        {
            'question': 'Do the preserved archives support a claim of stable benchmark-relative superiority by non-naive models?',
            'answer': 'No',
            'evidence': 'Naive0 remains the MAE leader at all three defended horizons in the preserved summary archive.',
        },
    ]
)

display(alignment_verdict)


,question,answer,evidence
0,"Does the current notebook stack already cover ETL, EDA, core modelling, and saved backtest outputs?",Yes,"The v1 ETL, integrity, modelling, and saved result archives are present in the repository."
1,Is another heavy modelling notebook required to address the dissertation purpose?,No,"The main weakness is fragmented reproducibility and stale result-analysis paths, not an absence of core benchmark evidence."
2,Is an additional consolidation notebook methodologically justified?,Yes,This notebook replaces the thesis-defence role that the legacy result-analysis notebook no longer fulfils cleanly.
3,Should the dissertation explicitly discuss realised feature-domain coverage?,Yes,"The preserved backtest feature matrix is overwhelmingly market/technical, so multimodal scope must be framed as partial rather than complete."
4,Do the preserved archives support a claim of stable benchmark-relative superiority by non-naive models?,No,Naive0 remains the MAE leader at all three defended horizons in the preserved summary archive.


## Next steps

- Use this notebook as the canonical thesis-defence bridge between the saved archives and the dissertation narrative.
- Retain the ETL, integrity, and backtesting notebooks, but avoid treating duplicate copies as equally authoritative.
- In the thesis text, state explicitly that the realised executable evidence is centred on market and technical predictors, with sentiment and network-related inputs documented as intended or partial coverage rather than as a fully preserved multimodal comparison.
- Do not broaden the empirical claims beyond what the preserved benchmark tables support.
